
# IJIES Technical Re-examination — Full Detection Rerun (Benchmark + Cross-domain)

This notebook performs a final, metric-consistent re-evaluation of the three detection models:

- YOLOv11m
- YOLOv12x
- RT-DETR-L

on both:

- benchmark test set: 17,400 images
- corrected cross-domain set: 870 images

The preserved Stage-1 YOLOv8 hand detector

`/kaggle/input/models/toilaxien/model-yolov8/tensorflow2/default/1/hand_yolov8s (1).pt`

is used only to create initial hand bounding-box proposals when verified bounding-box annotations are unavailable.

**Scientific safeguard:** YOLOv8 proposals are not automatically treated as ground truth. Precision, Recall, mAP50, and mAP50–95 are enabled only for datasets whose bounding boxes have been manually verified/corrected, or for which an existing verified `data.yaml` is supplied.


In [ ]:
# ============================================================
# 0. INSTALL / RUNTIME
# ============================================================

%pip install -q -U ultralytics pandas scikit-learn matplotlib pillow pyyaml

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
import gc
import json
import time
import hashlib
import shutil
import platform
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

print("Python:", sys.version)

gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
)

print("\nGPU inventory:")
print(gpu_info.stdout if gpu_info.stdout else "nvidia-smi unavailable")


In [ ]:
# ============================================================
# 1. CONFIG
# ============================================================

BENCHMARK_ROOT_HINT = Path(
    "/kaggle/input/datasets/toilaxien/asl-benchmark-test-split"
)

CROSS_DOMAIN_ROOT_HINT = Path(
    "/kaggle/input/datasets/toilaxien/asl-test-cross-domain-datase/cross_domain_dataset"
)

MODEL_ROOT = Path(
    "/kaggle/input/datasets/toilaxien/asl-models-realtime-test"
)

WORK_ROOT = Path(
    "/kaggle/working/IJIES_EDITOR_DETECTION_BENCHMARK_CROSSDOMAIN_WORK"
)

EVIDENCE_ROOT = Path(
    "/kaggle/working/IJIES_EDITOR_DETECTION_BENCHMARK_CROSSDOMAIN_EVIDENCE"
)

WORK_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_IMAGES = {
    "benchmark": 17400,
    "cross_domain": 870,
}

EXPECTED_CLASSES = 29

CLASS_LABELS = [
    "A","B","C","D","E","F","G","H","I","J","K","L","M","N",
    "O","P","Q","R","S","T","U","V","W","X","Y","Z",
    "delete","nothing","space",
]

MODEL_PATHS = {
    "YOLOv11m_det": MODEL_ROOT / "YoLo11m_dectection.pt",
    "YOLOv12x": MODEL_ROOT / "YoLo12x.pt",
    "RTDETR_L": MODEL_ROOT / "rtdetr-l.pt",
}

DETECTION_MODELS = [
    "YOLOv11m_det",
    "YOLOv12x",
    "RTDETR_L",
]

DISPLAY_NAMES = {
    "YOLOv11m_det": "YOLOv11m",
    "YOLOv12x": "YOLOv12x",
    "RTDETR_L": "RT-DETR-L",
}

# Memory-safe T4 configuration.
# Each detector runs in its own subprocess, so GPU memory is released
# before the next detector starts.
DETECTION_IMGSZ = 224

DETECTION_BATCH_SIZE = {
    "YOLOv11m_det": 4,
    "YOLOv12x": 1,
    "RTDETR_L": 1,
}

DETECTION_PRED_CONF = 0.001
DETECTION_PRED_IOU = 0.70

# Fresh re-test requested for the technical re-examination.
# Set False only when intentionally resuming cached detector predictions.
FORCE_RERUN = True

# ------------------------------------------------------------
# TRAINING-FAITHFUL DETECTOR TEST TARGETS
# ------------------------------------------------------------
# The original detector-training notebook created one YOLO target
# for EVERY image:
#
#     class_id 0.5 0.5 1.0 1.0
#
# Therefore the benchmark/cross-domain detector test must reconstruct
# exactly the same target representation.
#
# IMPORTANT:
# - `nothing` is a real detector class, not background.
# - `nothing` = detector class ID 27.
# - benchmark canonical `delete` maps to training class name `del`
#   at detector class ID 26.
# - YOLOv8 Stage-1 hand boxes are NOT used as GT for these 3 models.

DETECTOR_TRAIN_NAMES = [
    "A","B","C","D","E","F","G","H","I","J","K","L","M","N",
    "O","P","Q","R","S","T","U","V","W","X","Y","Z",
    "del","nothing","space",
]

DETECTOR_NAME_TO_ID = {
    name: i
    for i, name in enumerate(DETECTOR_TRAIN_NAMES)
}

NOTHING_LABEL = "nothing"

EXPECTED_NOTHING_IMAGES = {
    "benchmark": 600,
    "cross_domain": 30,
}

# Fresh bbox validation unless intentionally resuming.
FORCE_BBOX_RERUN = True

print("Benchmark :", BENCHMARK_ROOT_HINT)
print("Cross     :", CROSS_DOMAIN_ROOT_HINT)
print("Models    :", MODEL_ROOT)
print("Work      :", WORK_ROOT)


In [ ]:
# ============================================================
# 2. DATASET DISCOVERY + MANIFEST
# ============================================================

IMAGE_EXTS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

def canonical_label(name):
    s = str(name).strip()

    if len(s) == 1:
        return s.upper()

    low = s.lower()

    if low == "del":
        return "delete"

    if low in {"delete", "nothing", "space"}:
        return low

    return s

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()

def resolve_class_root(root_hint):
    root_hint = Path(root_hint)

    if not root_hint.exists():
        raise FileNotFoundError(root_hint)

    # Candidate 1: root itself contains the 29 class folders.
    candidates = [root_hint]

    # Candidate 2: nested folders (common Kaggle dataset layout).
    candidates.extend(
        p for p in root_hint.rglob("*")
        if p.is_dir()
    )

    expected = set(CLASS_LABELS)

    for candidate in candidates:
        child_dirs = [
            p for p in candidate.iterdir()
            if p.is_dir()
        ]

        child_labels = {
            canonical_label(p.name)
            for p in child_dirs
        }

        if expected.issubset(child_labels):
            return candidate

    raise RuntimeError(
        f"Could not locate a 29-class image root under: {root_hint}"
    )

def build_manifest(dataset_name, root_hint):
    class_root = resolve_class_root(root_hint)

    rows = []

    class_dirs = {
        canonical_label(p.name): p
        for p in class_root.iterdir()
        if p.is_dir()
    }

    for class_index, label in enumerate(CLASS_LABELS):
        class_dir = class_dirs.get(label)

        if class_dir is None:
            raise RuntimeError(
                f"{dataset_name}: missing class folder {label}"
            )

        images = sorted(
            p for p in class_dir.rglob("*")
            if p.is_file()
            and p.suffix.lower() in IMAGE_EXTS
        )

        for p in images:
            rows.append({
                "dataset": dataset_name,
                "relative_path": str(p.relative_to(class_root)),
                "absolute_path": str(p),
                "label": label,
                "true_index": class_index,
                "sha256": sha256_file(p),
            })

    df = pd.DataFrame(rows)

    expected_n = EXPECTED_IMAGES[dataset_name]

    if len(df) != expected_n:
        raise RuntimeError(
            f"{dataset_name}: expected {expected_n} images, "
            f"found {len(df)}."
        )

    if df["label"].nunique() != EXPECTED_CLASSES:
        raise RuntimeError(
            f"{dataset_name}: expected {EXPECTED_CLASSES} classes, "
            f"found {df['label'].nunique()}."
        )

    duplicate_groups = int(
        (df.groupby("sha256").size() > 1).sum()
    )

    audit = {
        "dataset": dataset_name,
        "resolved_class_root": str(class_root),
        "n_images": int(len(df)),
        "n_classes": int(df["label"].nunique()),
        "min_images_per_class": int(df.groupby("label").size().min()),
        "max_images_per_class": int(df.groupby("label").size().max()),
        "exact_duplicate_sha256_groups": duplicate_groups,
    }

    out_dir = WORK_ROOT / "datasets" / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    manifest_csv = out_dir / "manifest.csv"
    audit_json = out_dir / "audit.json"

    df.to_csv(manifest_csv, index=False)

    audit_json.write_text(
        json.dumps(audit, indent=2),
        encoding="utf-8",
    )

    print("\n", dataset_name, json.dumps(audit, indent=2))

    return {
        "root": class_root,
        "manifest": df,
        "manifest_csv": manifest_csv,
        "audit": audit,
    }

dataset_info = {
    "benchmark": build_manifest(
        "benchmark",
        BENCHMARK_ROOT_HINT,
    ),
    "cross_domain": build_manifest(
        "cross_domain",
        CROSS_DOMAIN_ROOT_HINT,
    ),
}

if dataset_info["cross_domain"]["audit"]["exact_duplicate_sha256_groups"] != 0:
    raise RuntimeError(
        "Cross-domain contains exact internal duplicate images."
    )

print("\n✅ Dataset audits passed.")


In [ ]:
# ============================================================
# 3. BENCHMARK ↔ CROSS-DOMAIN EXACT OVERLAP AUDIT
# ============================================================

benchmark_manifest = dataset_info["benchmark"]["manifest"]
cross_manifest = dataset_info["cross_domain"]["manifest"]

overlap = benchmark_manifest[
    ["relative_path", "label", "sha256"]
].merge(
    cross_manifest[
        ["relative_path", "label", "sha256"]
    ],
    on="sha256",
    how="inner",
    suffixes=("_benchmark", "_cross_domain"),
)

overlap_path = (
    WORK_ROOT
    / "benchmark_cross_exact_overlap.csv"
)

overlap.to_csv(
    overlap_path,
    index=False,
)

print("Exact benchmark ↔ cross-domain overlap pairs:", len(overlap))

if len(overlap) != 0:
    raise RuntimeError(
        f"Expected the corrected cross-domain set to have zero exact overlap, "
        f"but found {len(overlap)} pairs."
    )

print("✅ Exact overlap audit: PASS (0 pairs)")


In [ ]:
# ============================================================
# 4. DETECTOR CHECKPOINT INVENTORY
# ============================================================

def checkpoint_sha256(path):
    return sha256_file(path)

inventory_rows = []

for model_name in DETECTION_MODELS:
    model_path = MODEL_PATHS[model_name]

    if not model_path.exists():
        raise FileNotFoundError(
            f"Missing detector checkpoint: {model_path}"
        )

    inventory_rows.append({
        "model": model_name,
        "display_name": DISPLAY_NAMES[model_name],
        "checkpoint_filename": model_path.name,
        "checkpoint_bytes": int(model_path.stat().st_size),
        "checkpoint_sha256": checkpoint_sha256(model_path),
    })

model_inventory = pd.DataFrame(inventory_rows)

model_inventory.to_csv(
    WORK_ROOT / "detector_model_inventory.csv",
    index=False,
)

display(model_inventory)


# ------------------------------------------------------------
# Stage-1 YOLOv8 checkpoint inventory
# ------------------------------------------------------------
if not STAGE1_YOLOV8_PATH.exists():
    raise FileNotFoundError(
        f"Missing Stage-1 YOLOv8 checkpoint: {STAGE1_YOLOV8_PATH}"
    )

stage1_inventory = {
    "role": "Stage-1 hand bounding-box proposal generator",
    "checkpoint_filename": STAGE1_YOLOV8_PATH.name,
    "checkpoint_bytes": int(STAGE1_YOLOV8_PATH.stat().st_size),
    "checkpoint_sha256": checkpoint_sha256(STAGE1_YOLOV8_PATH),
    "imgsz": STAGE1_BBOX_IMGSZ,
    "confidence_threshold": STAGE1_BBOX_CONF,
    "iou_threshold": STAGE1_BBOX_IOU,
    "scientific_note": (
        "YOLOv8 outputs are initial bounding-box proposals only. "
        "They are not ground truth until manually reviewed/corrected."
    ),
}

(
    WORK_ROOT / "stage1_yolov8_checkpoint_inventory.json"
).write_text(
    json.dumps(stage1_inventory, indent=2),
    encoding="utf-8",
)

display(pd.DataFrame([stage1_inventory]))


## 5. Memory-safe detector worker

Each model runs in a separate Python subprocess. This prevents TensorFlow/PyTorch memory retained by another model from causing CUDA OOM. YOLOv12x and RT-DETR-L use batch size 1 on the T4.


In [ ]:
# ============================================================
# 5. WRITE MEMORY-SAFE DETECTOR WORKER
# ============================================================

DET_WORKER_PATH = (
    WORK_ROOT
    / "ijies_detection_imagelevel_worker.py"
)

DET_WORKER_PATH.write_text(
r"""
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from ultralytics import YOLO

try:
    from ultralytics import RTDETR
except Exception:
    RTDETR = None

MODEL_NAME = sys.argv[1]
MODEL_PATH = Path(sys.argv[2])
MANIFEST_CSV = Path(sys.argv[3])
OUT_DIR = Path(sys.argv[4])
IMGSZ = int(sys.argv[5])
BATCH_SIZE = int(sys.argv[6])
CONF = float(sys.argv[7])
IOU = float(sys.argv[8])

CLASS_LABELS = [
    "A","B","C","D","E","F","G","H","I","J","K","L","M","N",
    "O","P","Q","R","S","T","U","V","W","X","Y","Z",
    "delete","nothing","space",
]

def canonical_label(name):
    s = str(name).strip()

    if len(s) == 1:
        return s.upper()

    low = s.lower()

    if low == "del":
        return "delete"

    if low in {"delete", "nothing", "space"}:
        return low

    return s

manifest = pd.read_csv(MANIFEST_CSV)

OUT_DIR.mkdir(parents=True, exist_ok=True)

partial_csv = OUT_DIR / "predictions.partial.csv"
final_csv = OUT_DIR / "predictions.csv"
metadata_json = OUT_DIR / "worker_metadata.json"

rows = []
done_paths = set()

if partial_csv.exists():
    old = pd.read_csv(partial_csv)

    if "relative_path" in old.columns:
        rows = old.to_dict("records")
        done_paths = set(
            old["relative_path"].astype(str)
        )

        print(
            "Resume existing rows:",
            len(done_paths),
        )

remaining = manifest[
    ~manifest["relative_path"].astype(str).isin(done_paths)
].copy()

torch.cuda.empty_cache()
gc.collect()

load_t0 = time.perf_counter()

if MODEL_NAME == "RTDETR_L" and RTDETR is not None:
    try:
        model = RTDETR(str(MODEL_PATH))
    except Exception:
        model = YOLO(str(MODEL_PATH))
else:
    model = YOLO(str(MODEL_PATH))

load_seconds = time.perf_counter() - load_t0

raw_names = model.names

if isinstance(raw_names, dict):
    model_names = {
        int(k): canonical_label(v)
        for k, v in raw_names.items()
    }
else:
    model_names = {
        i: canonical_label(v)
        for i, v in enumerate(raw_names)
    }

dataset_name_map = {
    x.lower(): x
    for x in CLASS_LABELS
}

# Fail early if the detector class vocabulary does not match.
for model_label in model_names.values():
    if model_label.lower() not in dataset_name_map:
        raise RuntimeError(
            f"{MODEL_NAME}: detector class '{model_label}' "
            "does not map to the shared 29-class vocabulary."
        )

records = remaining.to_dict("records")

def save_partial():
    pd.DataFrame(rows).to_csv(
        partial_csv,
        index=False,
    )

for start in range(
    0,
    len(records),
    BATCH_SIZE,
):
    batch = records[
        start:start + BATCH_SIZE
    ]

    paths = [
        r["absolute_path"]
        for r in batch
    ]

    infer_t0 = time.perf_counter()

    results = model.predict(
        source=paths,
        imgsz=IMGSZ,
        batch=BATCH_SIZE,
        conf=CONF,
        iou=IOU,
        device=0,
        verbose=False,
        stream=False,
    )

    batch_elapsed_ms = (
        time.perf_counter()
        - infer_t0
    ) * 1000.0

    if len(results) != len(batch):
        raise RuntimeError(
            f"{MODEL_NAME}: expected {len(batch)} results, "
            f"received {len(results)}."
        )

    for source_row, result in zip(
        batch,
        results,
    ):
        true_label = canonical_label(
            source_row["label"]
        )

        pred_label = "__no_detection__"
        pred_index = -1
        confidence = 0.0
        n_boxes = 0

        boxes = getattr(
            result,
            "boxes",
            None,
        )

        if boxes is not None and len(boxes) > 0:
            n_boxes = int(len(boxes))

            confs = (
                boxes.conf
                .detach()
                .cpu()
                .numpy()
                .astype(float)
            )

            classes = (
                boxes.cls
                .detach()
                .cpu()
                .numpy()
                .astype(int)
            )

            best_i = int(
                np.argmax(confs)
            )

            raw_index = int(
                classes[best_i]
            )

            confidence = float(
                confs[best_i]
            )

            raw_label = canonical_label(
                model_names[raw_index]
            )

            mapped = dataset_name_map.get(
                raw_label.lower()
            )

            if mapped is None:
                raise RuntimeError(
                    f"{MODEL_NAME}: cannot map predicted label "
                    f"'{raw_label}'."
                )

            pred_label = mapped
            pred_index = CLASS_LABELS.index(
                pred_label
            )

        rows.append({
            "relative_path": source_row["relative_path"],
            "absolute_path": source_row["absolute_path"],
            "sha256": source_row["sha256"],
            "true_label": true_label,
            "true_index": int(source_row["true_index"]),
            "pred_label": pred_label,
            "pred_index": int(pred_index),
            "confidence": float(confidence),
            "n_boxes": int(n_boxes),
            "correct": int(pred_label == true_label),
            "batch_elapsed_ms": float(batch_elapsed_ms),
            "batch_mean_inference_ms_per_image": float(
                batch_elapsed_ms / max(1, len(batch))
            ),
        })

    save_partial()

    # Release result tensors before the next batch.
    del results

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    gc.collect()

    if (
        start == 0
        or
        (start + len(batch)) % 500 == 0
        or
        start + len(batch) == len(records)
    ):
        print(
            MODEL_NAME,
            min(start + len(batch), len(records)),
            "/",
            len(records),
        )

df = pd.DataFrame(rows)

if len(df) != len(manifest):
    raise RuntimeError(
        f"{MODEL_NAME}: expected {len(manifest)} prediction rows, "
        f"got {len(df)}."
    )

if df["relative_path"].nunique() != len(manifest):
    raise RuntimeError(
        f"{MODEL_NAME}: duplicate/missing prediction rows."
    )

df.to_csv(
    final_csv,
    index=False,
)

if partial_csv.exists():
    partial_csv.unlink()

metadata = {
    "model": MODEL_NAME,
    "checkpoint": MODEL_PATH.name,
    "n_predictions": int(len(df)),
    "imgsz": IMGSZ,
    "batch_size": BATCH_SIZE,
    "conf": CONF,
    "iou": IOU,
    "load_seconds": float(load_seconds),
    "no_detection_images": int(
        (df["pred_label"] == "__no_detection__").sum()
    ),
    "metric_scope": (
        "fresh detector image-level recognition inference; "
        "not bounding-box mAP"
    ),
}

metadata_json.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

del model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

gc.collect()

print("__PASS__")
print(json.dumps(metadata))
""",
    encoding="utf-8",
)

print("Worker:", DET_WORKER_PATH)


## 6. Sequential detector evaluation — benchmark + cross-domain

This is the main execution cell. It runs six fresh detector jobs in total:

- 3 detectors × benchmark;
- 3 detectors × cross-domain.

A completed `predictions.csv` is reused unless `FORCE_RERUN=True`.


In [ ]:
# ============================================================
# 6. RUN 3 DETECTORS ON BOTH DATASETS
# ============================================================

run_rows = []

def run_child(label, cmd, timeout):
    print("\n" + "=" * 90)
    print("START:", label)
    print("=" * 90)

    t0 = time.perf_counter()

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=timeout,
        env={
            **os.environ,
            "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
        },
    )

    elapsed = time.perf_counter() - t0

    print("returncode:", result.returncode)
    print("wall_seconds:", round(elapsed, 2))

    if result.stdout:
        print(result.stdout[-12000:])

    if result.stderr:
        print(result.stderr[-12000:])

    return {
        "returncode": int(result.returncode),
        "wall_seconds": float(elapsed),
    }

for dataset_name in [
    "benchmark",
    "cross_domain",
]:
    manifest_csv = dataset_info[
        dataset_name
    ]["manifest_csv"]

    n_images = len(
        dataset_info[
            dataset_name
        ]["manifest"]
    )

    print("\n\n" + "#" * 90)
    print(
        "DETECTION DATASET:",
        dataset_name,
        "| images:",
        n_images,
    )
    print("#" * 90)

    for model_name in DETECTION_MODELS:
        out_dir = (
            WORK_ROOT
            / "detection"
            / dataset_name
            / model_name
        )

        final_csv = (
            out_dir
            / "predictions.csv"
        )

        if FORCE_RERUN and out_dir.exists():
            shutil.rmtree(out_dir)

        if final_csv.exists():
            existing = pd.read_csv(final_csv)

            if len(existing) == n_images:
                print(
                    "SKIP completed:",
                    dataset_name,
                    model_name,
                )

                run_rows.append({
                    "dataset": dataset_name,
                    "model": model_name,
                    "status": "PASS_CACHED",
                    "returncode": 0,
                    "wall_seconds": None,
                })

                continue

        cmd = [
            sys.executable,
            str(DET_WORKER_PATH),
            model_name,
            str(MODEL_PATHS[model_name]),
            str(manifest_csv),
            str(out_dir),
            str(DETECTION_IMGSZ),
            str(DETECTION_BATCH_SIZE[model_name]),
            str(DETECTION_PRED_CONF),
            str(DETECTION_PRED_IOU),
        ]

        result = run_child(
            f"detection / {dataset_name} / {model_name}",
            cmd,
            timeout=8 * 60 * 60,
        )

        run_rows.append({
            "dataset": dataset_name,
            "model": model_name,
            "status": (
                "PASS"
                if result["returncode"] == 0
                else "FAILED"
            ),
            "returncode": result["returncode"],
            "wall_seconds": result["wall_seconds"],
        })

        # Parent process does not hold the detector, but clean anyway.
        gc.collect()

run_status = pd.DataFrame(run_rows)

run_status.to_csv(
    WORK_ROOT / "detection_worker_status.csv",
    index=False,
)

display(run_status)

failed = run_status[
    ~run_status["status"].isin(
        ["PASS", "PASS_CACHED"]
    )
]

if not failed.empty:
    print("\nFAILED DETECTOR RUNS:")
    display(failed)

    raise RuntimeError(
        "At least one detector run failed. "
        "Completed predictions are preserved and will be resumed next run."
    )

print("\n✅ All 3 detection models completed on benchmark + cross-domain.")


## 7. Detection image-level metrics and blue confusion matrices

For each detector and dataset, this section saves Accuracy, macro Precision, macro Recall, macro F1, raw predictions, classification report, and a blue confusion matrix.

These are **image-level detector recognition metrics**. They must not be labeled as mAP.


In [ ]:
# ============================================================
# 7. AGGREGATE DETECTOR IMAGE-LEVEL RESULTS
# ============================================================

summary_rows = []

for dataset_name in [
    "benchmark",
    "cross_domain",
]:
    for model_name in DETECTION_MODELS:
        out_dir = (
            WORK_ROOT
            / "detection"
            / dataset_name
            / model_name
        )

        pred_csv = (
            out_dir
            / "predictions.csv"
        )

        if not pred_csv.exists():
            raise FileNotFoundError(pred_csv)

        df = pd.read_csv(pred_csv)

        expected_n = EXPECTED_IMAGES[
            dataset_name
        ]

        if len(df) != expected_n:
            raise RuntimeError(
                f"{dataset_name}/{model_name}: expected "
                f"{expected_n} predictions, found {len(df)}."
            )

        y_true = (
            df["true_label"]
            .astype(str)
            .map(canonical_label)
        )

        y_pred = (
            df["pred_label"]
            .astype(str)
            .map(canonical_label)
        )

        # ----------------------------------------------------
        # CRITICAL NOTHING-CLASS INTEGRITY CHECK
        # ----------------------------------------------------
        # `nothing` is one of the 29 target classes in the paper.
        # It must NOT be dropped because Stage-1 YOLOv8 finds no hand.
        nothing_mask = (
            y_true == NOTHING_LABEL
        )

        nothing_count = int(
            nothing_mask.sum()
        )

        expected_nothing = EXPECTED_NOTHING_IMAGES[
            dataset_name
        ]

        if nothing_count != expected_nothing:
            raise RuntimeError(
                f"{dataset_name}/{model_name}: "
                f"`nothing` samples lost: "
                f"{nothing_count}/{expected_nothing}."
            )

        nothing_correct = int(
            (
                y_pred[nothing_mask]
                == NOTHING_LABEL
            ).sum()
        )

        nothing_accuracy = (
            nothing_correct
            / nothing_count
            if nothing_count
            else float("nan")
        )

        metrics = {
            "dataset": dataset_name,
            "model": model_name,
            "display_name": DISPLAY_NAMES[model_name],
            "n_images": int(len(df)),
            "no_detection_images": int(
                (y_pred == "__no_detection__").sum()
            ),
            "nothing_images_evaluated": nothing_count,
            "nothing_correct": nothing_correct,
            "nothing_accuracy": float(nothing_accuracy),
            "accuracy": float(
                accuracy_score(
                    y_true,
                    y_pred,
                )
            ),
            "macro_precision": float(
                precision_score(
                    y_true,
                    y_pred,
                    labels=CLASS_LABELS,
                    average="macro",
                    zero_division=0,
                )
            ),
            "macro_recall": float(
                recall_score(
                    y_true,
                    y_pred,
                    labels=CLASS_LABELS,
                    average="macro",
                    zero_division=0,
                )
            ),
            "macro_f1": float(
                f1_score(
                    y_true,
                    y_pred,
                    labels=CLASS_LABELS,
                    average="macro",
                    zero_division=0,
                )
            ),
            "metric_scope": (
                "image-level detector recognition; "
                "all 29 classes retained; no YOLOv8 filtering; "
                "not bbox mAP"
            ),
        }

        summary_rows.append(metrics)

        (
            out_dir
            / "imagelevel_metrics_summary.json"
        ).write_text(
            json.dumps(metrics, indent=2),
            encoding="utf-8",
        )

        report = classification_report(
            y_true,
            y_pred,
            labels=CLASS_LABELS,
            output_dict=True,
            zero_division=0,
        )

        pd.DataFrame(report).T.to_csv(
            out_dir
            / "classification_report.csv"
        )

        pred_labels = list(
            CLASS_LABELS
        )

        if (
            y_pred
            == "__no_detection__"
        ).any():
            pred_labels.append(
                "__no_detection__"
            )

        cm = np.zeros(
            (
                len(CLASS_LABELS),
                len(pred_labels),
            ),
            dtype=int,
        )

        true_map = {
            label: i
            for i, label in enumerate(
                CLASS_LABELS
            )
        }

        pred_map = {
            label: i
            for i, label in enumerate(
                pred_labels
            )
        }

        for true_label, pred_label in zip(
            y_true,
            y_pred,
        ):
            if (
                true_label in true_map
                and
                pred_label in pred_map
            ):
                cm[
                    true_map[true_label],
                    pred_map[pred_label],
                ] += 1

        pd.DataFrame(
            cm,
            index=CLASS_LABELS,
            columns=pred_labels,
        ).to_csv(
            out_dir
            / "confusion_matrix.csv"
        )

        fig, ax = plt.subplots(figsize=(13.2, 10.0))

        im = ax.imshow(
            cm,
            interpolation="nearest",
            cmap="Blues",
        )

        ax.set_title(DISPLAY_NAMES[model_name], fontsize=18, pad=8)

        ax.set_xlabel("Predicted", fontsize=14)
        ax.set_ylabel("True", fontsize=14)

        # Remove black border around the confusion matrix
        for spine in ax.spines.values():
            spine.set_visible(False)

        ax.tick_params(axis="both", which="both", length=0)

        ax.set_xticks(
            range(len(pred_labels))
        )

        ax.set_yticks(
            range(len(CLASS_LABELS))
        )

        ax.set_xticklabels(
            pred_labels,
            rotation=90,
            fontsize=12.5,
        )

        ax.set_yticklabels(
            CLASS_LABELS,
            fontsize=12.5,
        )

        threshold = (
            cm.max() / 2.0
            if cm.size
            else 0
        )

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                value = int(
                    cm[i, j]
                )

                ax.text(
                    j,
                    i,
                    str(value),
                    ha="center",
                    va="center",
                    fontsize=17,
                    color=(
                        "white"
                        if value > threshold
                        else "black"
                    ),
                )

        cbar = fig.colorbar(
            im,
            ax=ax,
            fraction=0.046,
            pad=0.04,
        )
        cbar.ax.tick_params(labelsize=12.5)
        for spine in cbar.ax.spines.values():
            spine.set_visible(False)

        fig.tight_layout(pad=0.6)

        fig_path = (
            out_dir
            / "confusion_matrix_blue.png"
        )

        fig.savefig(
            fig_path,
            dpi=300,
            bbox_inches="tight",
        )


        # Save an additional manuscript-facing filename for the
        # final cross-domain matrices.
        if dataset_name == "cross_domain":
            manuscript_name = {
                "YOLOv11m_det": "YOLOv11m_CrossDomain_ConfusionMatrix.png",
                "YOLOv12x": "Figure5_YOLOv12x_CrossDomain_Final.png",
                "RTDETR_L": "RTDETR_L_CrossDomain_ConfusionMatrix.png",
            }[model_name]

            fig.savefig(
                WORK_ROOT / manuscript_name,
                dpi=300,
                bbox_inches="tight",
            )

        # Show manuscript-relevant cross-domain matrices inline.
        if dataset_name == "cross_domain":
            plt.show()

        plt.close(fig)

detection_summary = pd.DataFrame(
    summary_rows
)

detection_summary.to_csv(
    WORK_ROOT
    / "detection_imagelevel_summary_benchmark_cross.csv",
    index=False,
)

display(detection_summary)


In [ ]:
# ============================================================
# 8. BENCHMARK → CROSS-DOMAIN IMAGE-LEVEL DEGRADATION
# ============================================================

degradation_rows = []

for model_name in DETECTION_MODELS:
    benchmark_row = detection_summary[
        (detection_summary["dataset"] == "benchmark")
        &
        (detection_summary["model"] == model_name)
    ].iloc[0]

    cross_row = detection_summary[
        (detection_summary["dataset"] == "cross_domain")
        &
        (detection_summary["model"] == model_name)
    ].iloc[0]

    degradation_rows.append({
        "model": model_name,
        "display_name": DISPLAY_NAMES[model_name],
        "metric": "image_level_accuracy",
        "benchmark": float(benchmark_row["accuracy"]),
        "cross_domain": float(cross_row["accuracy"]),
        "drop_pp": 100.0 * (
            float(benchmark_row["accuracy"])
            -
            float(cross_row["accuracy"])
        ),
        "note": (
            "Supporting detector recognition comparison; "
            "not a replacement for mAP50 degradation."
        ),
    })

detection_degradation = pd.DataFrame(
    degradation_rows
)

detection_degradation.to_csv(
    WORK_ROOT
    / "detection_imagelevel_benchmark_to_cross_degradation.csv",
    index=False,
)

display(detection_degradation)


## 9. Reconstruct the detector ground truth exactly as in training

The detector-training notebooks use a classification-like detection representation:
each image contains exactly one object whose bounding box spans the full image.

For every benchmark and cross-domain image, the target is therefore:

```text
class_id 0.5 0.5 1.0 1.0
```

Class mapping is kept identical to detector training:

```text
A-Z     -> 0-25
del     -> 26
nothing -> 27
space   -> 28
```

The current test dataset uses canonical label `delete`; this is mapped back to training
label `del` before writing YOLO ground truth.

`nothing` is never represented by an empty label and is never treated as background.
YOLOv8 Stage-1 is not used to create these ground-truth labels.


In [ ]:
# ============================================================
# 9A. BUILD TRAINING-FAITHFUL FULL-IMAGE YOLO GROUND TRUTH
# ============================================================

TRAINING_FAITHFUL_GT_ROOT = (
    WORK_ROOT
    / "training_faithful_fullimage_gt"
)

def canonical_to_detector_name(label):
    label = canonical_label(label)

    if label == "delete":
        return "del"

    return label


def safe_link_or_copy(src, dst):
    src = Path(src)
    dst = Path(dst)

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if dst.exists() or dst.is_symlink():
        return

    try:
        os.symlink(
            str(src),
            str(dst),
        )
    except Exception:
        shutil.copy2(
            src,
            dst,
        )


def build_training_faithful_detection_gt(
    dataset_name
):
    manifest_df = dataset_info[
        dataset_name
    ]["manifest"].copy()

    out_root = (
        TRAINING_FAITHFUL_GT_ROOT
        / dataset_name
    )

    if (
        FORCE_BBOX_RERUN
        and
        out_root.exists()
    ):
        shutil.rmtree(
            out_root
        )

    images_root = (
        out_root
        / "images"
        / "val"
    )

    labels_root = (
        out_root
        / "labels"
        / "val"
    )

    images_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    labels_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    audit_rows = []

    for row in manifest_df.to_dict(
        "records"
    ):
        rel = Path(
            row["relative_path"]
        )

        src_img = Path(
            row["absolute_path"]
        )

        dst_img = (
            images_root
            / rel
        )

        dst_label = (
            labels_root
            / rel.with_suffix(
                ".txt"
            )
        )

        safe_link_or_copy(
            src_img,
            dst_img,
        )

        dst_label.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        canonical = canonical_label(
            row["label"]
        )

        detector_name = (
            canonical_to_detector_name(
                canonical
            )
        )

        if detector_name not in DETECTOR_NAME_TO_ID:
            raise RuntimeError(
                f"{dataset_name}: unknown class mapping "
                f"{canonical} -> {detector_name}"
            )

        class_id = (
            DETECTOR_NAME_TO_ID[
                detector_name
            ]
        )

        # EXACT target construction from detector training:
        # class_id x_center y_center width height
        dst_label.write_text(
            f"{class_id} 0.5 0.5 1.0 1.0\n",
            encoding="utf-8",
        )

        audit_rows.append({
            "relative_path": str(
                rel
            ),
            "canonical_label": canonical,
            "detector_name": detector_name,
            "class_id": int(
                class_id
            ),
            "x_center": 0.5,
            "y_center": 0.5,
            "width": 1.0,
            "height": 1.0,
        })

    audit_df = pd.DataFrame(
        audit_rows
    )

    expected_total = EXPECTED_IMAGES[
        dataset_name
    ]

    if len(audit_df) != expected_total:
        raise RuntimeError(
            f"{dataset_name}: expected "
            f"{expected_total} GT rows, "
            f"found {len(audit_df)}."
        )

    # --------------------------------------------------------
    # `nothing` must be a normal target class.
    # --------------------------------------------------------
    nothing_df = audit_df[
        audit_df[
            "canonical_label"
        ] == NOTHING_LABEL
    ]

    expected_nothing = EXPECTED_NOTHING_IMAGES[
        dataset_name
    ]

    if len(nothing_df) != expected_nothing:
        raise RuntimeError(
            f"{dataset_name}: expected "
            f"{expected_nothing} `nothing` targets, "
            f"found {len(nothing_df)}."
        )

    if not (
        nothing_df[
            "class_id"
        ] == 27
    ).all():
        raise RuntimeError(
            f"{dataset_name}: `nothing` must be class ID 27."
        )

    # `delete` in current manifests must map to detector `del` / ID 26.
    delete_df = audit_df[
        audit_df[
            "canonical_label"
        ] == "delete"
    ]

    if len(delete_df) > 0:
        if not (
            (
                delete_df[
                    "detector_name"
                ] == "del"
            )
            &
            (
                delete_df[
                    "class_id"
                ] == 26
            )
        ).all():
            raise RuntimeError(
                f"{dataset_name}: delete -> del / class 26 mapping failed."
            )

    # No empty YOLO labels are allowed.
    empty_labels = [
        p
        for p in labels_root.rglob(
            "*.txt"
        )
        if not p.read_text(
            encoding="utf-8"
        ).strip()
    ]

    if empty_labels:
        raise RuntimeError(
            f"{dataset_name}: "
            f"{len(empty_labels)} empty GT label files found."
        )

    # Current Ultralytics validates both train and val keys.
    # This notebook calls model.val(split="val") only;
    # no training is performed.
    data_yaml = {
        "path": str(
            out_root
        ),
        "train": "images/val",
        "val": "images/val",
        "nc": len(
            DETECTOR_TRAIN_NAMES
        ),
        "names": {
            i: name
            for i, name
            in enumerate(
                DETECTOR_TRAIN_NAMES
            )
        },
    }

    yaml_path = (
        out_root
        / "data.yaml"
    )

    with yaml_path.open(
        "w",
        encoding="utf-8",
    ) as f:
        yaml.safe_dump(
            data_yaml,
            f,
            sort_keys=False,
            allow_unicode=True,
        )

    audit_df.to_csv(
        out_root
        / "fullimage_gt_manifest.csv",
        index=False,
    )

    audit = {
        "dataset": dataset_name,
        "n_images": int(
            len(audit_df)
        ),
        "n_classes": int(
            audit_df[
                "class_id"
            ].nunique()
        ),
        "nothing_images": int(
            len(nothing_df)
        ),
        "nothing_class_id": 27,
        "delete_training_name": "del",
        "delete_class_id": 26,
        "target_box": [
            0.5,
            0.5,
            1.0,
            1.0,
        ],
        "empty_label_files": 0,
        "training_protocol_match": True,
    }

    (
        out_root
        / "fullimage_gt_audit.json"
    ).write_text(
        json.dumps(
            audit,
            indent=2,
        ),
        encoding="utf-8",
    )

    print(
        "\n✅ TRAINING-FAITHFUL GT:",
        dataset_name,
    )

    print(
        json.dumps(
            audit,
            indent=2,
        )
    )

    return {
        "root": out_root,
        "yaml": yaml_path,
        "audit_df": audit_df,
        "audit": audit,
    }


TRAINING_FAITHFUL_GT = {
    dataset_name: (
        build_training_faithful_detection_gt(
            dataset_name
        )
    )
    for dataset_name in [
        "benchmark",
        "cross_domain",
    ]
}

TRAINING_FAITHFUL_DATA_YAML = {
    dataset_name: info[
        "yaml"
    ]
    for dataset_name, info
    in TRAINING_FAITHFUL_GT.items()
}

print(
    "\n✅ Training-faithful detector GT created for both datasets."
)


### 9B. Ground-truth integrity

Before running bbox metrics, the notebook verifies:

- 17,400 benchmark targets and 870 cross-domain targets;
- every target has a full-image box `0.5 0.5 1.0 1.0`;
- no empty labels;
- all 600 benchmark `nothing` images and all 30 cross-domain `nothing` images remain;
- `nothing` is class ID 27;
- canonical `delete` maps to training class `del`, ID 26.


In [ ]:
# ============================================================
# 9B. VERIFY TRAINING-FAITHFUL GT BEFORE model.val()
# ============================================================

gt_integrity_rows = []

for dataset_name in [
    "benchmark",
    "cross_domain",
]:
    info = TRAINING_FAITHFUL_GT[
        dataset_name
    ]

    df = info[
        "audit_df"
    ]

    expected_total = EXPECTED_IMAGES[
        dataset_name
    ]

    expected_nothing = EXPECTED_NOTHING_IMAGES[
        dataset_name
    ]

    checks = {
        "dataset": dataset_name,
        "expected_images": expected_total,
        "gt_rows": int(
            len(df)
        ),
        "nothing_expected": expected_nothing,
        "nothing_gt_rows": int(
            (
                df[
                    "canonical_label"
                ] == "nothing"
            ).sum()
        ),
        "nothing_class27": bool(
            (
                df.loc[
                    df[
                        "canonical_label"
                    ] == "nothing",
                    "class_id",
                ]
                == 27
            ).all()
        ),
        "delete_maps_to_del26": bool(
            (
                (
                    df.loc[
                        df[
                            "canonical_label"
                        ] == "delete",
                        "detector_name",
                    ]
                    == "del"
                )
                &
                (
                    df.loc[
                        df[
                            "canonical_label"
                        ] == "delete",
                        "class_id",
                    ]
                    == 26
                )
            ).all()
        ),
        "all_boxes_full_image": bool(
            (
                (df["x_center"] == 0.5)
                &
                (df["y_center"] == 0.5)
                &
                (df["width"] == 1.0)
                &
                (df["height"] == 1.0)
            ).all()
        ),
    }

    if checks["gt_rows"] != expected_total:
        raise RuntimeError(
            f"{dataset_name}: GT row count mismatch."
        )

    if checks["nothing_gt_rows"] != expected_nothing:
        raise RuntimeError(
            f"{dataset_name}: `nothing` GT count mismatch."
        )

    if not checks["nothing_class27"]:
        raise RuntimeError(
            f"{dataset_name}: `nothing` is not class ID 27."
        )

    if not checks["delete_maps_to_del26"]:
        raise RuntimeError(
            f"{dataset_name}: delete -> del / ID26 mapping failed."
        )

    if not checks["all_boxes_full_image"]:
        raise RuntimeError(
            f"{dataset_name}: GT bbox differs from training protocol."
        )

    gt_integrity_rows.append(
        checks
    )


gt_integrity = pd.DataFrame(
    gt_integrity_rows
)

gt_integrity.to_csv(
    WORK_ROOT
    / "training_faithful_gt_integrity.csv",
    index=False,
)

display(
    gt_integrity
)

print(
    "\n✅ GT integrity PASS."
)


## 10. Training-faithful bbox Precision / Recall / mAP50 / mAP50–95

Each detector is validated against the same full-image target construction used during training.
This replaces the previous YOLOv8 hand-box pseudo-GT experiment.


In [ ]:
# ============================================================
# 10. TRAINING-FAITHFUL BBOX P/R / mAP50 / mAP50-95
# ============================================================

MAP_WORKER_PATH = (
    WORK_ROOT
    / "ijies_training_faithful_map_worker.py"
)

MAP_WORKER_PATH.write_text(
r"""
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys
import json
import gc
from pathlib import Path

import torch
from ultralytics import YOLO

try:
    from ultralytics import RTDETR
except Exception:
    RTDETR = None

MODEL_NAME = sys.argv[1]
MODEL_PATH = Path(sys.argv[2])
DATA_YAML = Path(sys.argv[3])
DATASET_NAME = sys.argv[4]
IMGSZ = int(sys.argv[5])
BATCH_SIZE = int(sys.argv[6])
OUT_DIR = Path(sys.argv[7])

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

torch.cuda.empty_cache()
gc.collect()

if MODEL_NAME == "RTDETR_L" and RTDETR is not None:
    try:
        model = RTDETR(
            str(MODEL_PATH)
        )
    except Exception:
        model = YOLO(
            str(MODEL_PATH)
        )
else:
    model = YOLO(
        str(MODEL_PATH)
    )

metrics = model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    conf=0.001,
    iou=0.70,
    device=0,
    plots=True,
    verbose=False,
    project=str(OUT_DIR),
    name="ultralytics_val",
    exist_ok=True,
)

box = metrics.box

summary = {
    "dataset": DATASET_NAME,
    "model": MODEL_NAME,
    "precision": float(box.mp),
    "recall": float(box.mr),
    "mAP50": float(box.map50),
    "mAP50_95": float(box.map),
    "imgsz": IMGSZ,
    "batch_size": BATCH_SIZE,
    "conf": 0.001,
    "iou": 0.70,
    "gt_protocol": (
        "training-faithful full-image target "
        "class_id 0.5 0.5 1.0 1.0"
    ),
    "nothing_class_id": 27,
}

(
    OUT_DIR
    / "metrics_summary.json"
).write_text(
    json.dumps(
        summary,
        indent=2,
    ),
    encoding="utf-8",
)

print("__PASS__")
print(json.dumps(summary))
""",
    encoding="utf-8",
)


map_status_rows = []

for dataset_name in [
    "benchmark",
    "cross_domain",
]:
    data_yaml = (
        TRAINING_FAITHFUL_DATA_YAML[
            dataset_name
        ]
    )

    for model_name in DETECTION_MODELS:
        out_dir = (
            WORK_ROOT
            / "detection_map"
            / dataset_name
            / model_name
        )

        if (
            FORCE_BBOX_RERUN
            and
            out_dir.exists()
        ):
            shutil.rmtree(
                out_dir
            )

        metrics_json = (
            out_dir
            / "metrics_summary.json"
        )

        if (
            metrics_json.exists()
            and
            not FORCE_BBOX_RERUN
        ):
            print(
                "SKIP existing mAP:",
                dataset_name,
                model_name,
            )

            map_status_rows.append({
                "dataset": dataset_name,
                "model": model_name,
                "status": "PASS_CACHED",
            })

            continue

        result = run_child(
            f"training-faithful mAP / "
            f"{dataset_name} / {model_name}",
            [
                sys.executable,
                str(MAP_WORKER_PATH),
                model_name,
                str(
                    MODEL_PATHS[
                        model_name
                    ]
                ),
                str(data_yaml),
                dataset_name,
                str(DETECTION_IMGSZ),
                str(
                    DETECTION_BATCH_SIZE[
                        model_name
                    ]
                ),
                str(out_dir),
            ],
            timeout=8 * 60 * 60,
        )

        status = (
            "PASS"
            if result[
                "returncode"
            ] == 0
            else "FAILED"
        )

        map_status_rows.append({
            "dataset": dataset_name,
            "model": model_name,
            "status": status,
        })

        if status != "PASS":
            raise RuntimeError(
                f"mAP failed: "
                f"{dataset_name}/{model_name}"
            )


map_status = pd.DataFrame(
    map_status_rows
)

map_status.to_csv(
    WORK_ROOT
    / "detection_map_status.csv",
    index=False,
)

display(
    map_status
)


# ------------------------------------------------------------
# AGGREGATE BBOX METRICS
# ------------------------------------------------------------

map_summary_rows = []

for dataset_name in [
    "benchmark",
    "cross_domain",
]:
    for model_name in DETECTION_MODELS:
        p = (
            WORK_ROOT
            / "detection_map"
            / dataset_name
            / model_name
            / "metrics_summary.json"
        )

        if not p.exists():
            raise FileNotFoundError(
                p
            )

        row = json.loads(
            p.read_text(
                encoding="utf-8"
            )
        )

        row[
            "display_name"
        ] = DISPLAY_NAMES[
            model_name
        ]

        row[
            "status"
        ] = "PASS"

        map_summary_rows.append(
            row
        )


detection_map_summary = pd.DataFrame(
    map_summary_rows
)

detection_map_summary.to_csv(
    WORK_ROOT
    / "detection_bbox_map_summary.csv",
    index=False,
)

display(
    detection_map_summary[
        [
            "dataset",
            "display_name",
            "precision",
            "recall",
            "mAP50",
            "mAP50_95",
        ]
    ]
)


# ------------------------------------------------------------
# TABLE 3
# ------------------------------------------------------------

benchmark_map = (
    detection_map_summary[
        detection_map_summary[
            "dataset"
        ] == "benchmark"
    ][
        [
            "display_name",
            "precision",
            "recall",
            "mAP50",
            "mAP50_95",
        ]
    ]
    .rename(
        columns={
            "display_name": "model"
        }
    )
    .copy()
)

benchmark_map.to_csv(
    WORK_ROOT
    / "Table3_detection_benchmark_rerun.csv",
    index=False,
)


# ------------------------------------------------------------
# TABLE 4
# ------------------------------------------------------------

cross_map = (
    detection_map_summary[
        detection_map_summary[
            "dataset"
        ] == "cross_domain"
    ][
        [
            "display_name",
            "precision",
            "recall",
            "mAP50",
            "mAP50_95",
        ]
    ]
    .rename(
        columns={
            "display_name": "model"
        }
    )
    .copy()
)

cross_map.to_csv(
    WORK_ROOT
    / "Table4_detection_crossdomain_rerun.csv",
    index=False,
)


# ------------------------------------------------------------
# TABLE 7
# ------------------------------------------------------------

table7_rerun = (
    benchmark_map[
        [
            "model",
            "mAP50",
        ]
    ]
    .rename(
        columns={
            "mAP50": "benchmark_mAP50"
        }
    )
    .merge(
        cross_map[
            [
                "model",
                "mAP50",
            ]
        ].rename(
            columns={
                "mAP50": "cross_domain_mAP50"
            }
        ),
        on="model",
        how="inner",
    )
)

table7_rerun[
    "drop_pp"
] = (
    table7_rerun[
        "benchmark_mAP50"
    ]
    -
    table7_rerun[
        "cross_domain_mAP50"
    ]
) * 100.0

table7_rerun.to_csv(
    WORK_ROOT
    / "Table7_detection_degradation_rerun.csv",
    index=False,
)

print(
    "\n=== TABLE 3 — BENCHMARK ==="
)
display(
    benchmark_map
)

print(
    "\n=== TABLE 4 — CROSS-DOMAIN ==="
)
display(
    cross_map
)

print(
    "\n=== TABLE 7 — mAP50 DEGRADATION ==="
)
display(
    table7_rerun
)

print(
    "\nMean detection mAP50 degradation =",
    f"{table7_rerun['drop_pp'].mean():.2f} pp"
)


In [ ]:
# ============================================================
# 11. BUILD DETECTION-ONLY EDITOR EVIDENCE PACKAGE
# ============================================================

if EVIDENCE_ROOT.exists():
    shutil.rmtree(
        EVIDENCE_ROOT
    )

EVIDENCE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

def sanitize_csv(
    src_path,
    dst_path,
):
    src_path = Path(
        src_path
    )

    dst_path = Path(
        dst_path
    )

    dst_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    df = pd.read_csv(
        src_path
    )

    for c in [
        "absolute_path",
        "path",
    ]:
        if c in df.columns:
            df = df.drop(
                columns=[
                    c
                ]
            )

    df.to_csv(
        dst_path,
        index=False,
    )


# ------------------------------------------------------------
# A. Protocol
# ------------------------------------------------------------

protocol = {
    "purpose": (
        "IJIES post-publication technical "
        "re-examination — final 3-detector test"
    ),
    "evaluation_scope": (
        "YOLOv11m, YOLOv12x, RT-DETR-L"
    ),
    "models": [
        DISPLAY_NAMES[
            model_name
        ]
        for model_name in DETECTION_MODELS
    ],
    "benchmark_images": EXPECTED_IMAGES[
        "benchmark"
    ],
    "cross_domain_images": EXPECTED_IMAGES[
        "cross_domain"
    ],
    "canonical_test_classes": CLASS_LABELS,
    "detector_training_names": (
        DETECTOR_TRAIN_NAMES
    ),
    "detector_training_class_map": {
        str(i): name
        for i, name
        in enumerate(
            DETECTOR_TRAIN_NAMES
        )
    },
    "critical_gt_definition": (
        "Every detector-training image was assigned "
        "one YOLO target: class_id 0.5 0.5 1.0 1.0. "
        "The final re-evaluation reconstructs the "
        "same full-image target for every benchmark "
        "and cross-domain image."
    ),
    "nothing_policy": (
        "`nothing` is class ID 27 and receives "
        "the same full-image box. It is not background "
        "and no empty GT label is created."
    ),
    "delete_mapping": (
        "Canonical test label `delete` maps to "
        "detector training label `del`, class ID 26."
    ),
    "yolov8_stage1_role": (
        "Not used to construct detector benchmark or "
        "cross-domain ground truth."
    ),
    "imgsz": DETECTION_IMGSZ,
    "prediction_confidence_threshold": (
        DETECTION_PRED_CONF
    ),
    "prediction_iou_threshold": (
        DETECTION_PRED_IOU
    ),
    "batch_sizes": (
        DETECTION_BATCH_SIZE
    ),
    "metric_note": (
        "Image-level Accuracy/Precision/Recall/F1 are "
        "supporting detector-recognition metrics. "
        "Bounding-box Precision/Recall/mAP50/mAP50-95 "
        "are evaluated against the training-faithful "
        "full-image target representation."
    ),
}

(
    EVIDENCE_ROOT
    / "environment_and_protocol.json"
).write_text(
    json.dumps(
        protocol,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# B. Paper-ready tables + summaries
# ------------------------------------------------------------

for name in [
    "detection_imagelevel_summary_benchmark_cross.csv",
    "detection_imagelevel_benchmark_to_cross_degradation.csv",
    "detection_bbox_map_summary.csv",
    "detection_map_status.csv",
    "training_faithful_gt_integrity.csv",
    "Table3_detection_benchmark_rerun.csv",
    "Table4_detection_crossdomain_rerun.csv",
    "Table7_detection_degradation_rerun.csv",
]:
    p = (
        WORK_ROOT
        / name
    )

    if p.exists():
        sanitize_csv(
            p,
            EVIDENCE_ROOT
            / name,
        )


# ------------------------------------------------------------
# C. Dataset manifests / audits
# ------------------------------------------------------------

for dataset_name in [
    "benchmark",
    "cross_domain",
]:
    src_dir = (
        WORK_ROOT
        / "datasets"
        / dataset_name
    )

    dst_dir = (
        EVIDENCE_ROOT
        / "datasets"
        / dataset_name
    )

    dst_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    for p in src_dir.iterdir():
        if p.suffix.lower() == ".csv":
            sanitize_csv(
                p,
                dst_dir
                / p.name,
            )

        elif p.suffix.lower() == ".json":
            data = json.loads(
                p.read_text(
                    encoding="utf-8"
                )
            )

            data.pop(
                "resolved_class_root",
                None,
            )

            (
                dst_dir
                / p.name
            ).write_text(
                json.dumps(
                    data,
                    indent=2,
                ),
                encoding="utf-8",
            )


# ------------------------------------------------------------
# D. Training-faithful GT evidence
# ------------------------------------------------------------

for dataset_name in [
    "benchmark",
    "cross_domain",
]:
    gt_root = (
        TRAINING_FAITHFUL_GT[
            dataset_name
        ][
            "root"
        ]
    )

    dst_root = (
        EVIDENCE_ROOT
        / "training_faithful_gt"
        / dataset_name
    )

    dst_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    for name in [
        "data.yaml",
        "fullimage_gt_manifest.csv",
        "fullimage_gt_audit.json",
    ]:
        p = (
            gt_root
            / name
        )

        if not p.exists():
            continue

        if p.suffix.lower() == ".csv":
            sanitize_csv(
                p,
                dst_root
                / name,
            )
        else:
            shutil.copy2(
                p,
                dst_root
                / name,
            )


# ------------------------------------------------------------
# E. Image-level per-model outputs
# ------------------------------------------------------------

for dataset_name in [
    "benchmark",
    "cross_domain",
]:
    for model_name in DETECTION_MODELS:
        src_dir = (
            WORK_ROOT
            / "detection"
            / dataset_name
            / model_name
        )

        if not src_dir.exists():
            continue

        dst_dir = (
            EVIDENCE_ROOT
            / "imagelevel_detection"
            / dataset_name
            / DISPLAY_NAMES[
                model_name
            ]
        )

        dst_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        for p in src_dir.iterdir():
            if not p.is_file():
                continue

            if p.suffix.lower() == ".csv":
                sanitize_csv(
                    p,
                    dst_dir
                    / p.name,
                )

            elif p.suffix.lower() in {
                ".json",
                ".png",
            }:
                shutil.copy2(
                    p,
                    dst_dir
                    / p.name,
                )


# ------------------------------------------------------------
# F. Raw bbox-validation outputs / plots
# ------------------------------------------------------------

map_root = (
    WORK_ROOT
    / "detection_map"
)

if map_root.exists():
    shutil.copytree(
        map_root,
        EVIDENCE_ROOT
        / "bbox_validation",
        dirs_exist_ok=True,
        symlinks=False,
    )


# ------------------------------------------------------------
# G. Code
# ------------------------------------------------------------

code_dir = (
    EVIDENCE_ROOT
    / "code"
)

code_dir.mkdir(
    parents=True,
    exist_ok=True,
)

for p in [
    DET_WORKER_PATH,
    MAP_WORKER_PATH,
]:
    if p.exists():
        shutil.copy2(
            p,
            code_dir
            / p.name,
        )


# ------------------------------------------------------------
# H. README
# ------------------------------------------------------------

readme = """
# IJIES Final 3-Detector Re-evaluation Evidence

Models:
- YOLOv11m
- YOLOv12x
- RT-DETR-L

Final evaluation sets:
- benchmark: 17,400 images
- corrected cross-domain: 870 images
- 29 classes

Training-faithful detector target
---------------------------------
The original detector-training notebook assigns every image exactly one
YOLO target:

    class_id 0.5 0.5 1.0 1.0

The final re-evaluation reconstructs exactly that target representation.

Important class handling:
- A-Z: IDs 0-25
- canonical `delete` -> detector training `del`: ID 26
- `nothing`: ID 27
- `space`: ID 28

`nothing` is never background and never receives an empty label.

Stage-1 YOLOv8 hand boxes are not used as ground truth for these three
detector benchmark/cross-domain metrics.

Primary outputs:
- Table3_detection_benchmark_rerun.csv
- Table4_detection_crossdomain_rerun.csv
- Table7_detection_degradation_rerun.csv
- detection_bbox_map_summary.csv
- detection_imagelevel_summary_benchmark_cross.csv
"""

(
    EVIDENCE_ROOT
    / "README.md"
).write_text(
    readme.strip()
    + "\n",
    encoding="utf-8",
)


archive = shutil.make_archive(
    "/kaggle/working/"
    "IJIES_FINAL_3DETECTOR_TRAINING_FAITHFUL_EVIDENCE",
    "zip",
    root_dir=str(
        EVIDENCE_ROOT
    ),
)

print(
    "✅ Detection editor evidence ZIP:",
    archive,
)

print(
    "✅ Table 3:",
    WORK_ROOT
    / "Table3_detection_benchmark_rerun.csv",
)

print(
    "✅ Table 4:",
    WORK_ROOT
    / "Table4_detection_crossdomain_rerun.csv",
)

print(
    "✅ Table 7:",
    WORK_ROOT
    / "Table7_detection_degradation_rerun.csv",
)


In [ ]:
# ============================================================
# 12. FINAL TRAINING-FAITHFUL TEST INTEGRITY CHECK
# ============================================================

for dataset_name in [
    "benchmark",
    "cross_domain",
]:
    expected_total = EXPECTED_IMAGES[
        dataset_name
    ]

    expected_nothing = EXPECTED_NOTHING_IMAGES[
        dataset_name
    ]

    # Image-level detector results
    for model_name in DETECTION_MODELS:
        row = detection_summary[
            (
                detection_summary[
                    "dataset"
                ] == dataset_name
            )
            &
            (
                detection_summary[
                    "model"
                ] == model_name
            )
        ]

        if len(row) != 1:
            raise RuntimeError(
                f"Missing image-level row: "
                f"{dataset_name}/{model_name}"
            )

        row = row.iloc[
            0
        ]

        if int(
            row[
                "n_images"
            ]
        ) != expected_total:
            raise RuntimeError(
                f"{dataset_name}/{model_name}: "
                "not all images were evaluated."
            )

        if int(
            row[
                "nothing_images_evaluated"
            ]
        ) != expected_nothing:
            raise RuntimeError(
                f"{dataset_name}/{model_name}: "
                "`nothing` class was not fully retained."
            )

    # Ground-truth construction
    gt_df = TRAINING_FAITHFUL_GT[
        dataset_name
    ][
        "audit_df"
    ]

    if len(gt_df) != expected_total:
        raise RuntimeError(
            f"{dataset_name}: GT count mismatch."
        )

    nothing_gt = gt_df[
        gt_df[
            "canonical_label"
        ] == "nothing"
    ]

    if len(nothing_gt) != expected_nothing:
        raise RuntimeError(
            f"{dataset_name}: `nothing` GT count mismatch."
        )

    if not (
        nothing_gt[
            "class_id"
        ] == 27
    ).all():
        raise RuntimeError(
            f"{dataset_name}: `nothing` is not class 27."
        )

    delete_gt = gt_df[
        gt_df[
            "canonical_label"
        ] == "delete"
    ]

    if not (
        (
            delete_gt[
                "detector_name"
            ] == "del"
        )
        &
        (
            delete_gt[
                "class_id"
            ] == 26
        )
    ).all():
        raise RuntimeError(
            f"{dataset_name}: delete mapping mismatch."
        )

    if not (
        (
            (gt_df["x_center"] == 0.5)
            &
            (gt_df["y_center"] == 0.5)
            &
            (gt_df["width"] == 1.0)
            &
            (gt_df["height"] == 1.0)
        ).all()
    ):
        raise RuntimeError(
            f"{dataset_name}: GT boxes do not match training."
        )


if len(
    detection_map_summary
) != 6:
    raise RuntimeError(
        "Expected six bbox metric rows."
    )

if not (
    detection_map_summary[
        "status"
    ] == "PASS"
).all():
    raise RuntimeError(
        "At least one bbox metric run failed."
    )

print(
    "✅ 6/6 image-level detector tests complete."
)

print(
    "✅ 6/6 training-faithful bbox evaluations complete."
)

print(
    "✅ Benchmark retains all 600 `nothing` images per model."
)

print(
    "✅ Cross-domain retains all 30 `nothing` images per model."
)

print(
    "✅ `nothing` = class ID 27 with full-image GT."
)

print(
    "✅ `delete` -> training class `del` = ID 26."
)

print(
    "✅ Every detector GT box = 0.5 0.5 1.0 1.0."
)

print(
    "✅ No YOLOv8 hand box is used as detector GT."
)

print(
    "✅ Final training-faithful detector test complete."
)
